# Análsis de la copa mundial femenina (Parte II)

En el presente notebook se realizará la transformación de los datos generados en el proceso anterior con el fin de extraer características relevantes para el análisis.

## Importación de las librerias

In [ ]:
import json
import os
import sys


import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


In [ ]:
DATALAKEHOUSE_DIR = "./datalakehouse/"
DUCKDB_FILE = os.path.join(DATALAKEHOUSE_DIR, "datalakehouse.duckdb")

## Carga de los datos de entrada desde duckdb

Los datos de entrada para esta esta del workflow provienen del equema "s01_curated" de la base de datos duckdb con la que se ha venido trabajando en el notebook anterior.

Se generan dos dataframes: 
    - **df_wcm**: contiene la información de los partidos jugados en la copa mundial femenina.
    - **df_wcw**: contiene la información resumida de los resultados del mundial. 


In [ ]:
df_wcw = None
df_wcm = None

with duckdb.connect(DUCKDB_FILE) as conn:
    df_wcw = conn.execute("SELECT * FROM s01_curated.world_cup_women;").df()
    df_wcm = conn.execute("SELECT * FROM s01_curated.world_cup_matches;").df()

## construccion de la tabla 5

**df_intermediate** Se ha decido construir una tabla con la información de los partidos por equipo. Esta tabla toma la información de **df_matches** y genera dos registros, uno para cada equipo riva y en la tabla almacena el número de goles, el score, los penals, las tarjetas, etc. 


La tabla intermedia también se almacena dentro de la base de datos duckdb para su uso en el análisis.


In [ ]:
def winner_team(row):
    if row['home_score'] > row['away_score']:
        return 'home'
    elif row['home_score'] < row['away_score']:
        return 'away'
    elif pd.isna(row['home_penalty']) and pd.isna(row['away_penalty']):
        return 'draw'
    elif row['home_penalty'] > row['away_penalty']:
        return 'home'
    elif row['home_penalty'] < row['away_penalty']:
        return 'away'
    return 'draw'


def generate_intermediate_table(df_matches: pd.DataFrame) -> pd.DataFrame:

    output_records = []

    for i, row in df_matches.iterrows():
        
        winner = winner_team(row)

        temp_record = {
            'MatchID': row['id'],
            'Year': row['Year'],
            'Date': row['Date'],
            'Host': row['Host'],
            'Attendance': row['Attendance'],
            'Venue': row['Venue'],
            'Round': row['Round'],
        }

        home_record = temp_record.copy()
        home_record.update({
            'Team': row['home_team'],
            'Type': 'Home',
            'Opponent': row['away_team'],

            'Score': row['home_score'],
            'Score_penalties': row['home_penalty'],
            'Against': row['away_score'],
            'Against_penalties': row['away_penalty'],
            'Game_result': 'Win' if winner == 'home' else ('Loss' if winner == 'away' else 'Draw'),
            'Assistences': row['home_goal_long'].count('Assist') if row['home_goal_long'] else 0,

            'yellow_card': (row['home_yellow_card_long'].count(',') + 1) if row['home_yellow_card_long'] else 0,
            'yellow_red_card': (row['home_yellow_red_card'].count(',') + 1) if row['home_yellow_red_card'] else 0,
            'red_card': (row['home_red_card'].count(',') + 1) if row['home_red_card'] else 0,
        })

        output_records.append(home_record)

        away_record = temp_record.copy()
        away_record.update({
            'Team': row['away_team'],
            'Type': 'Away',
            'Opponent': row['home_team'],

            'Score': row['away_score'],
            'Score_penalties': row['away_penalty'],
            'Against': row['home_score'],
            'Against_penalties': row['home_penalty'],
            'Game_result': 'Win' if winner == 'away' else ('Loss' if winner == 'home' else 'Draw'),
            'Assistences': row['away_goal_long'].count('Assist') if row['away_goal_long'] else 0,


            'yellow_card': (row['away_yellow_card_long'].count(',') + 1) if row['away_yellow_card_long'] else 0,
            'yellow_red_card': (row['away_yellow_red_card'].count(',') + 1) if row['away_yellow_red_card'] else 0,
            'red_card': (row['away_red_card'].count(',') + 1) if row['away_red_card'] else 0,
        })
        output_records.append(away_record)

    df_intermediate = pd.DataFrame.from_records(output_records)
    return df_intermediate

In [ ]:
df_intermediate = generate_intermediate_table(df_wcm)

In [ ]:
df_intermediate.head()

In [ ]:
with duckdb.connect(DUCKDB_FILE) as conn:
    conn.execute("CREATE SCHEMA IF NOT EXISTS s02_insights;")
    conn.execute("DROP TABLE IF EXISTS s02_insights.world_cup_women_goals;")
    conn.execute("DROP TABLE IF EXISTS s02_insights.world_cup_women_matches;")
    conn.execute("DROP TABLE IF EXISTS s02_insights.world_cup_women_top_players;")
    conn.execute("DROP TABLE IF EXISTS s02_insights.world_cup_women_teams_1991;")
    conn.execute("DROP TABLE IF EXISTS s02_insights.world_cup_women_teams_summary;")

In [ ]:
# save intermediate table as world_cup_women_matches
with duckdb.connect(DUCKDB_FILE) as conn:
    conn.execute("CREATE TABLE s02_insights.world_cup_women_matches AS SELECT * FROM df_intermediate;")
    print("Saved table s02_insights.world_cup_women_matches.")
    

#### Tabla 3

Elabore la tabla de posiciones del mundial realizado en 1991. Tenga en cuenta que cada partido ganado da 3 puntos, cada parti do
empatado da 1 punto. Así mismo, las tarjetas amarillas suman -1 punto para juego limpio y las tarjetas rojas – 2 puntos a juego
limpio.

La tabla debe tener la siguiente estructura:

| Equipo | Partidos Jugados (PJ) | Partidos Ganados (PG) | Partidos Empatados (PE) | Partidos Perdidos (PP)| Goles a Favor (GF) | Goles en Contra (GC) | Diferencia de Goles (GF – GC) | Juego Limpio (JL) | Puntos |
|--------|-----------------------|-----------------------|-------------------------|-----------------------|--------------------|----------------------|-------------------------------|-------------------|--------|
|        |                       |                       |                         |                       |                    |                      |                               |                   |        |



Esta tabla es generada a partir de la tabla intermedia. Se hace una agrupación por equipo y se calculan lo sigueinte:
    - **juegos jugados (PJ):** conteo de todos los partidos jugados por el equipo.  
    - **partidos ganados (PG):** conteo de todos los partidos ganados por el equipo.  
    - **partidos empatados (PE):** conteo de todos los partidos empatados por el equipo.  
    - **partidos perdidos (PP):** conteo de todos los partidos perdidos por el equipo.  
    - **goles a favor (GF):** conteo de todos los goles marcados por el equipo.  
    - **goles en contra (GC):** conteo de todos los goles recibidos por el equipo.  
    - **diferencia de goles (GF – GC):** resta de los goles a favor menos los goles en   contra.
    - **juego limpio (JL):** conteo de todos los partidos jugados por el equipo.  
    - **puntos:** conteo de todos los partidos jugados por el equipo.  

In [ ]:
def fair_play(row):
    return (row['Yellow_Cards'] * -1) + (row['Yellow_Red_Cards'] * -1) + (row['Red_Cards'] * -2)

def fair_play_alternative(row):
    return (row['Yellow_Cards'] * -1) + (row['Yellow_Red_Cards'] * -3) + (row['Red_Cards'] * -4)

def generate_tabla_3(df_intermediate: pd.DataFrame) -> pd.DataFrame:
    # Your implementation here
    df_matches_1991 = df_intermediate[df_intermediate['Year'] == 1991]
    df_grouped = df_matches_1991.groupby('Team').agg(
        Matches_Played=('MatchID', 'count'),
        Wins=('Game_result', lambda x: (x == 'Win').sum()),
        Draws=('Game_result', lambda x: (x == 'Draw').sum()),
        Losses=('Game_result', lambda x: (x == 'Loss').sum()),
        Goals_Scored=('Score', 'sum'),
        Goals_Conceded=('Against', 'sum'),
        Yellow_Cards=('yellow_card', 'sum'),
        Yellow_Red_Cards=('yellow_red_card', 'sum'),
        Red_Cards=('red_card', 'sum')
    )
    df_grouped.reset_index(inplace=True)
    df_grouped['Goal_Difference'] = df_grouped['Goals_Scored'] - df_grouped['Goals_Conceded']
    df_grouped['Fair_Play_Points'] = df_grouped.apply(fair_play, axis=1)
    df_grouped['Points'] = df_grouped['Wins'] * 3 + df_grouped['Draws'] * 1
    df_grouped = df_grouped.sort_values(
        by=['Points', 'Goal_Difference', 'Goals_Scored', 'Goals_Conceded', 'Fair_Play_Points'],
        ascending=[False, False, False, True, False]
    ).reset_index(drop=True)
    return df_grouped

df_tabla_3 = generate_tabla_3(df_intermediate)
df_tabla_3

In [ ]:
# save tabla_3 as world_cup_women_teams_1991
with duckdb.connect(DUCKDB_FILE) as conn:
    conn.execute("CREATE TABLE s02_insights.world_cup_women_teams_1991 AS SELECT * FROM df_tabla_3;")
    print("Saved table s02_insights.world_cup_women_teams_1991.")

#### Tabla de goles

A partir de la tabla `world_cup_matches` se ha decidido crear una tabla de goles.
Los campos principales de esta tabla de goles son: 

- `match_id`: Identificador único de cada partido.
- `team_id`: Identificador único de cada equipo.
- `player_id`: Identificador único de cada jugador.
- `minute`: Minuto en el que se anota el gol.
- `is_penalty`: Indica si el gol es un penalti (1) o no (0).
- `is_own_goal`: Indica si el gol es un gol propio (1) o no (0).


La tabla de goles se almacena dentro de la base de datos duckdb, y posteriormente se hace la tabla de goleadoras a partir de esa tabla.

In [ ]:
def goal_data(row, column, team, against, type_goal, temp_record):
    output_records = []
    for item in row[column].split('|'):
        player = item.split('·')[0].strip()
        minute = item.split('·')[1].strip()[:-1]
        record = {
            **temp_record,
            'Team': row[team],
            'Player': player,
            'Minute': minute,
            'Team_against': row[against],
            'Type_goal': type_goal
        }
        output_records.append(record)
    return output_records


def generate_goals_table(df_matches: pd.DataFrame) -> pd.DataFrame:
    
    output_records = []
    for i, row in df_matches.iterrows():
        
        temp_record = {
            'MatchID': row['id'],
            'Year': row['Year'],
            'Date': row['Date']
        }

        # goals
        if not pd.isna(row['home_goal']):
            home_goal_records = goal_data(row, 'home_goal', 'home_team', 'away_team', 'goal', temp_record)
            output_records.extend(home_goal_records)
        if not pd.isna(row['away_goal']):
            away_goal_records = goal_data(row, 'away_goal', 'away_team', 'home_team', 'goal', temp_record)
            output_records.extend(away_goal_records)
        
        # own
        if not pd.isna(row['home_own_goal']):
            home_own_records = goal_data(row, 'home_own_goal', 'home_team', 'away_team', 'own', temp_record)
            output_records.extend(home_own_records)
        if not pd.isna(row['away_own_goal']):
            away_own_records = goal_data(row, 'away_own_goal', 'away_team', 'home_team', 'own', temp_record)
            output_records.extend(away_own_records)

        # penalty
        if not pd.isna(row['home_penalty_goal']):
            home_goal_records = goal_data(row, 'home_penalty_goal', 'home_team', 'away_team', 'penalty', temp_record)
            output_records.extend(home_goal_records)
        if not pd.isna(row['away_penalty_goal']):
            away_goal_records = goal_data(row, 'away_penalty_goal', 'away_team', 'home_team', 'penalty', temp_record)
            output_records.extend(away_goal_records)

    df_goals = pd.DataFrame.from_records(output_records)
    df_goals['Player'] = df_goals['Player'].str.replace('(P)', '').str.replace('(OG)', '').str.strip()
    return df_goals

df_goals = generate_goals_table(df_wcm)
df_goals.head()

In [ ]:
# save goals table 
with duckdb.connect(DUCKDB_FILE) as conn:
    conn.execute("CREATE TABLE s02_insights.world_cup_women_goals AS SELECT * FROM df_goals;")
    print("Saved table s02_insights.world_cup_women_goals.")

La pabla de goleadores se contruye como una agrupación de la tabla de goles. 
Se hace una agrupación por temporada, equipo y jugador.

In [ ]:
def generate_table_4(
        df_goals: pd.DataFrame, 
        year: int = 2023,
        top: int = 10,
        include_penalty: bool = True) -> pd.DataFrame:

    df_goals_2023 = df_goals[df_goals['Year'] == year]
    df_goals_grouped = df_goals_2023.groupby(by=['Player', 'Team']).agg(
        Goals=('Type_goal', lambda x: (x == 'goal').sum()),
        Penalty_Goals=('Type_goal', lambda x: (x == 'penalty').sum()),
    )
    df_goals_grouped.reset_index(inplace=True)

    if not include_penalty:
        df_goals_grouped['Goals_Score'] = df_goals_grouped['Goals']
    else:
        df_goals_grouped['Goals_Score'] = df_goals_grouped['Goals'] + df_goals_grouped['Penalty_Goals']
    
    del df_goals_grouped['Goals']
    del df_goals_grouped['Penalty_Goals']

    if top:
        top_n_goals = df_goals_grouped['Goals_Score'].nlargest(top).min()
        df_goals_grouped = df_goals_grouped[df_goals_grouped['Goals_Score'] >= top_n_goals]

    df_goals_grouped = df_goals_grouped.sort_values(
        by=['Goals_Score'], ascending=[False]
    ).reset_index(drop=True)

    return df_goals_grouped


In [ ]:
df_table_4 = generate_table_4(df_goals, year=2023, top=10, include_penalty=True)
df_table_4

In [ ]:
# save table 4
with duckdb.connect(DUCKDB_FILE) as conn:
    conn.execute("CREATE TABLE s02_insights.world_cup_women_top_players AS SELECT * FROM df_table_4;")
    print("Saved table s02_insights.world_cup_women_top_players.")

#### Tabla 5. 


| Año | Host | Equipo | Partidos Jugados | Goles Totales marcados | Promedio de Goles marcados | Goles Totales recibidos | promedio de goles recibidos | partidos totales ganados | partidos totales perdidos | partidos totales empatados | promedio de asistencia por equipo |
|-----|------|--------|------------------|------------------------|----------------------------|-------------------------|-----------------------------|--------------------------|---------------------------|----------------------------|-----------------------------------|
|     |      |        |                  |                        |                            |                         |                             |                          |                           |                            |                                   |

In [ ]:
def generate_tabla_5(df_intermediate: pd.DataFrame) -> pd.DataFrame:
    
    df_grouped = df_intermediate.groupby(by=['Team', 'Host', 'Year']).agg(
        Matches_Played=('MatchID', 'count'),
        Total_Goals_Scored=('Score', 'sum'),
        Total_Goals_Conceded=('Against', 'sum'),
        Total_Attendance=('Attendance', 'sum'),
        Total_Wins=('Game_result', lambda x: (x == 'Win').sum()),
        Total_Losses=('Game_result', lambda x: (x == 'Loss').sum()),
        Total_Draws=('Game_result', lambda x: (x == 'Draw').sum()),
    )
    df_grouped.reset_index(inplace=True)
    df_grouped['Average_Goals_Scored'] = df_grouped['Total_Goals_Scored'] / df_grouped['Matches_Played']
    df_grouped['Average_Goals_Conceded'] = df_grouped['Total_Goals_Conceded'] / df_grouped['Matches_Played']
    df_grouped['Average_Attendance_Per_Team'] = df_grouped['Total_Attendance'] / df_grouped['Matches_Played']

    df_grouped = df_grouped.sort_values(
        by=['Year', 'Team'], ascending=[True, True]
    ).reset_index(drop=True)

    return df_grouped
    

In [ ]:
df_tabla_5 = generate_tabla_5(df_intermediate)
df_tabla_5

In [ ]:
with duckdb.connect(DUCKDB_FILE) as conn:
    conn.execute("CREATE TABLE s02_insights.world_cup_women_teams_summary AS SELECT * FROM df_tabla_5;")
    print("Saved table s02_insights.world_cup_women_teams_summary.")

In [ ]:
df_tabla_5.info()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Supón que tu DataFrame se llama 'df'
# df = ... carga tu DataFrame aquí ...

# --- Código para generar el diagrama de cajas ---

plt.figure(figsize=(14, 8)) # Ajusta el tamaño de la figura para que los años se vean mejor

# Usamos sns.boxplot para crear el gráfico
sns.boxplot(data=df_tabla_5, 
            x='Year',             # Eje X: Variable categórica (años)
            y='Average_Goals_Scored', # Eje Y: Variable numérica (promedio de goles)
            palette='viridis')    # Paleta de colores atractiva

# Mejorar la presentación del gráfico
plt.title('Distribución del Promedio de Goles Anotados por Partido a lo largo de los Años del Torneo', fontsize=16)
plt.xlabel('Año del Torneo', fontsize=12)
plt.ylabel('Promedio de Goles por Partido', fontsize=12)
plt.xticks(rotation=45) # Rota las etiquetas del eje X para que no se superpongan
plt.grid(True, axis='y', linestyle='--', alpha=0.6)

# Muestra el gráfico
plt.show()




Existe una tendencia a la "baja" porque en la medida que los equipos "mejoran" se hace más dificil marcadores con altaas tasas de goles. 

In [ ]:
# Calcular los puntos totales para cada fila (equipo/temporada)
df  = df_tabla_5.copy()

df['Total_Points'] = (df['Total_Wins'] * 3) + (df['Total_Draws'] * 1) + (df['Total_Losses'] * 0)

# Calcular los puntos promedio por partido (PPG)
df['Points_Per_Game'] = df['Total_Points'] / df['Matches_Played']

# Verificar las nuevas columnas (opcional)
print("DataFrame con nuevas columnas:")
print(df[['Team', 'Year', 'Total_Points', 'Points_Per_Game']].head())



plt.figure(figsize=(15, 9)) # Aumentar el tamaño para mejor visibilidad

# Crear el gráfico de líneas
sns.lineplot(data=df, 
             x='Year', 
             y='Points_Per_Game', 
             hue='Team',           # Diferencia cada línea por el nombre del equipo
             marker='o')           # Añade puntos a las líneas para mayor claridad

# Mejorar la presentación
plt.title('Puntos Promedio por Partido (PPG) para Cada Equipo a lo largo de los Años', fontsize=16)
plt.xlabel('Año del Torneo', fontsize=12)
plt.ylabel('Puntos Promedio por Partido (PPG)', fontsize=12)
plt.xticks(rotation=45)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left') # Mueve la leyenda fuera del gráfico
plt.grid(True, linestyle='--', alpha=0.6)

# Mostrar el gráfico
plt.show()

In [ ]:
import plotly.express as px

# Supón que tu DataFrame se llama 'df' y ya tiene la columna 'Points_Per_Game'
# df = ... carga tu DataFrame aquí y calcula PPG ...

# Si no has calculado PPG, usa este código:
# df['Total_Points'] = (df['Total_Wins'] * 3) + (df['Total_Draws'] * 1)
# df['Points_Per_Game'] = df['Total_Points'] / df['Matches_Played']


# --- Código para generar el gráfico interactivo ---

fig = px.line(df, 
              x='Year', 
              y='Points_Per_Game', 
              color='Team',        # Diferencia cada línea por el nombre del equipo
              markers=True,        # Añade marcadores interactivos en los puntos de datos
              title='Puntos Promedio por Partido (PPG) Interactivo por Equipo y Año')

# Personalizar el layout (opcional)
fig.update_layout(
    xaxis_title="Año del Torneo",
    yaxis_title="Puntos Promedio por Partido (PPG)",
    legend_title="Equipo"
)

# Mostrar el gráfico interactivo
# Esto renderizará un widget interactivo en la salida de tu notebook
fig.show()